# A short introduction to containerized software

After spending time using nf-core pipelines to answer bioinformatic questions, we will focus on the processes that lie behind these pipelines now.

Today, we will focus on containerization, namely via Docker. 



1. Check if Docker is installed.

In [ ]:
!docker info

### What is a container?


> A container is software packaging code and it's dependencies. It includes everything necessary to run an application: code, runtime, system tools/libraries and settings. 

### Why do we use containers?

> Containers are useful, since you don't have to download tools on your own system, but on a subsystem that only runs when needed. Also by using pre-specified images of containers, this can make sure that the right versions of software are used to guarantee reproducability & functionalty. Also containers have a fixed operating systems, so compliance of tools is given. Tools that won't run on e.g. mac, can run on the container which uses linux.

### What is a docker image?

>A Docker image is an executable, that can create a container. It image contains all the libraries and dependencies to make the container run. Since this is a file, you can share it to others to make it possible for them to use your environment, to make your work e.g. a bioinfo pipeline reproducable for others.

### Let's run our first docker image:

### Login to docker

In [ ]:
# This you need to do on the command line directly
!docker login   # I registered to docker hub

### Run your first docker container

In [ ]:
!docker run hello-world

```bash
# Hello from Docker!
# This message shows that your installation appears to be working correctly.
# [...]
```

### Find the container ID

In [ ]:
!docker ps -a # lists all existing containers. After that i searched for most recent one (hello world). see partial output below

```bash
f63b5e0d525b  hello-world    "/hello" 5 minutes ago   Exited (0) 5 minutes ago    vibrant_roentgen

### Delete the container again, give prove its deleted

In [ ]:
!docker rm f63b5e0d525b # command to delete it

> Deleting this echoes the id again, then i used  ```!docker ps -a``` again to see wether the hello-world image is still there, and it isn't.

### FASTQC is a very useful tool as you've learned last week. Let's try and run it from command line

Link to the software: https://www.bioinformatics.babraham.ac.uk/projects/fastqc/

Please describe the steps you took to download and run the software for the example fastq file from last week below:

1. I used ```conda install fastqc``` to install fastqc from conda. 
2. I used ```fastqc --noextract -o fastqc_results_no_container ../day_02/fetchngs_out/fetchngs_out/fastq/SRX19144486_SRR23195516_1.fastq.gz``` to run fastqc on one of the files from day 2.


### Very well, now let's try to make use of its docker container

1. create a container holding fastqc using seqera containers (https://seqera.io/containers/)
2. use the container to generate a fastqc html of the example fastq file

In [ ]:
# pull the container
!docker pull community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae
# This downloaded the image

In [ ]:
# run the container and save the results to a new "fastqc_results" directory
!docker run \
    --rm \
    -it \
    -v "$(pwd)/..:/curr" \
    community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae \
    fastqc \
    --noextract \
    -o /curr/day_04/fastqc_results \
    curr/day_02/fetchngs_out/fetchngs_out/fastq/SRX19144486_SRR23195516_1.fastq.gz

```bash
docker run \
    --rm \  # delete container afterwards
    -it \   # makes interactive shell
    -v "$(pwd)/..:/curr" \    # give a location from my machine to container, that contains data&outdir under it
    community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae \  # the container from seqera
    fastqc \                                                # run fastqc
    --noextract \                                           # don't extract output files (cleaner)
    -o /out /data/SRX19144486_SRR23195516_1.fastq.gz \      # add output directory
    /data/SRX19144486_SRR23195516_1.fastq.gz                # give fastqc the data
```

### Now that you know how to use a docker container, which approach was easier and which approach will be easier in the future?

> For one run the first approach is simpler, but if you need to run it in a pipeline as well as seperately versions&compatibility are important. Also the container is set up once, so future runs wont require this step. This makes the results easier and more reproducable for future runs. 

### What would you say, which approach is more reproducible?

> The one in the container, since the fastqc version & the operating system are fixed there, so with the same image someone else would get the same results.

### Compare the file to last weeks fastqc results, are they identical?
### Is the fastqc version identical?

> As far as it seems, the results are identical. This can be assumed by comparing plots inbetween the earlier run from within the rnaseq pipeline with the run my container produced:

**My container:**

![self_built](./images/container_self_built.png)

**RNASEQ:**

![rnaseq](./images/pipeline.png)

> Those two look identical to me, which strongly suggests the results to be identical.
> Further both, my and rnaseq's version used fastqc version 0.12.1, as can be read in the MultiQC report. For my own container, I know the version through testing (fastqc --version).

## Dockerfiles

We now used Docker containers and images directly to boost our research. 

Let's create our own toy Dockerfile including the "cowsay" tool (https://en.wikipedia.org/wiki/Cowsay)

Hints:
1. Docker is Linux, so you need to know the apt-get command to install "cowsay"

In [ ]:
# open the file "my_dockerfile" in a text editor

# RUN apt-get update && apt-get install -y curl && apt-get install -y cowsay
# ENV PATH="${PATH}:/usr/games"

### Explain the RUN and ENV lines you added to the file

> *RUN* executes installation commands upon building the image, so downloads cowsay and curl after updating. -y streamlines downloading by saying yes to any dialogue

> *ENV* adds cowsay’s directory to PATH, to make cowsay executable directly without typing the path, just by calling cowsay command after running the image

In [ ]:
# build the docker image
!docker build -f my_dockerfile -t my-cowsay .

In [ ]:
# make sure that the image has been built
!docker image ls
# prints list containing:
# my-cowsay:latest  a52e0eeba21a  225MB   59.1MB     
# So this worked.   

In [ ]:
# run the docker file 
!docker run --rm my-cowsay cowsay "42"

# works, prints a cow saying 42 as expected

## Let's do some bioinformatics with the docker file and create a new docker file that holds the salmon tool used in rnaseq

To do so, use "curl" in your new dockerfile to get salmon from https://github.com/COMBINE-lab/salmon/releases/download/v1.5.2/salmon-1.5.2_linux_x86_64.tar.gz

In [ ]:
# use the file "salmon_docker" in this directory to build a new docker image

In [ ]:
# build the image
!docker build -f salmon_docker -t salmon-custom .

In [ ]:
# run the docker image to give out the version of salmon
!docker run --rm salmon-custom salmon --version

#-> 1.5.2

## Do you think bioinformaticians have to create a docker image every time they want to run a tool?
> No, this would be unneccessary extra work for everyone, since images only need to be created once for a tool, if done well. You could then use those existing images.

Find the salmon docker image online and run it on your computer.

> I found this salmon container online: https://hub.docker.com/r/getwilds/salmon
> and pulled&ran the image with:

```bash
docker pull getwilds/salmon:latest
docker run --rm getwilds/salmon:latest salmon --version
```


What is https://biocontainers.pro/ ?
> It's a community project for building and hosting images for bio(-info) software. This makes life easier, since there often is a prebuild image for a tool, so one can just use this instead of creating an own one with all dependencies.


## Are there other ways to create Docker (or Apptainer) images?

What is https://seqera.io/containers/ ?

> Another way to create images is using seqera, which searches for tools in conda, adds them to an environment and adds the env to conda inside an container, making the tools usable as well. 